# NLP Lab-Week 1: Text Foundations
---

## Lab Roadmap

| # | Topic |
|---|-------
| 0 | Setup |
| 1 | Tokenization |
| 2 | Stemming & Lemmatization |
| 3 | Stop Words & Cleaning |
| 4 | Bag of Words & N-grams |
| 5 | TF-IDF |
| 6 | String Similarity |
---

> **Lab Philosophy:** Each section has **three layers**:
> - **Concept**: What it is and why it matters
> - **Starter code**:  heavily commented, **READ** before running
> - **Your turn**: fill-in cells; hints always provided and some reflection questions to deepen understanding
>
> 🟢 = Beginner-friendly | 🟡 = Intermediate | 🔴 = Challenging / Exciting   
> Cells marked `✍️` require your input as code **or** written response

---
## Section 0: Setup & Library Check

Run Cell A (install) then Cell B (import). If you already ran this notebook, both cells finish in < 30 s. If it is your **first time** doing the installations, it will take longer, especially for **Cell B**.

In [23]:
# Cell A -- install required Python packages (run once)

# Import built-in Python modules
import subprocess  # allows Python to run terminal/command-line commands
import sys         # provides access to the current Python interpreter

# List of external libraries required for this notebook/project
LIBS = [
    "nltk",          # classic NLP toolkit for tokenisation, stemming, corpora, etc.
    "spacy",         # industrial-strength NLP framework for advanced text processing
    "scikit-learn",  # machine learning utilities, vectorisers, and evaluation tools
    "transformers",  # Hugging Face library for transformer-based language models #Not done on 22nd.
    "Levenshtein",   # fast implementation of edit-distance calculations
    "tabulate",      # formats tables into clean readable text output
    "pandas",        # data manipulation and DataFrame operations
    "numpy",         # numerical computing and multidimensional arrays
]

# Loop through each library name in the list
for lib in LIBS:

    # Install the package using pip from inside the notebook
    # sys.executable ensures installation uses the same Python environment
    # currently running the notebook/kernel
    subprocess.check_call([
        sys.executable,  # current Python interpreter
        "-m",            # run library module as a script
        "pip",           # Python package installer
        "install",       # pip command to install packages
        lib,             # package name from the LIBS list
        "-q"             # quiet mode: reduces unnecessary installation logs
    ])

    print(f"{lib} installed successfully.")
print("All required packages have been installed successfully.")


nltk installed successfully.
spacy installed successfully.
scikit-learn installed successfully.
transformers installed successfully.
Levenshtein installed successfully.
tabulate installed successfully.
pandas installed successfully.
numpy installed successfully.
All required packages have been installed successfully.


In [24]:
# Cell B -- import libraries and download required NLP resources
# =========================
# Built-in Python modules
# =========================
import re        # regular expressions for text pattern matching
import math      # mathematical functions and operations
import string    # common string constants and utilities
import warnings  # controls warning messages shown in the notebook
# Hide unnecessary warning messages to keep notebook output clean
warnings.filterwarnings("ignore")
# =========================
# Numerical and data analysis libraries
# =========================
import numpy as np   # numerical arrays and mathematical operations
import pandas as pd  # DataFrame manipulation and structured data analysis
# =========================
# NLTK: Natural Language Toolkit
# =========================
import nltk
# Download required NLTK resources
# quiet=True suppresses verbose download logs
for resource in [

    # Sentence and word tokenisation models
    "punkt",
    "punkt_tab",

    # Common English stopwords
    "stopwords",

    # Lexical database used for lemmatisation
    "wordnet",

    # POS (part-of-speech) tagging models
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng"

]:
    nltk.download(resource, quiet=True)
    
# Import tokenisation functions
from nltk.tokenize import (
    word_tokenize,  # splits text into words/tokens
    sent_tokenize   # splits text into sentences
)

# Import stemming and lemmatisation tools
from nltk.stem import (
    PorterStemmer,      # classic stemming algorithm
    SnowballStemmer,    # improved multilingual stemming
    WordNetLemmatizer   # converts words into dictionary base forms
)

# Import English stopwords corpus
from nltk.corpus import stopwords as nltk_stopwords

# =========================
# spaCy: advanced NLP framework
# =========================

import spacy

# Try loading the small English spaCy model
# If the model is not installed, download it automatically

try:

    # Load pre-trained English NLP pipeline
    nlp = spacy.load("en_core_web_sm")

except OSError:

    # Download missing spaCy model using command line
    subprocess.run(
        ["python", "-m", "spacy", "download", "en_core_web_sm"],
        capture_output=True  # hides long installation logs
    )

    # Load the model again after installation
    nlp = spacy.load("en_core_web_sm")

# =========================
# scikit-learn: vectorisation and similarity
# =========================

from sklearn.feature_extraction.text import (

    # Converts text into raw token-frequency vectors
    CountVectorizer,

    # Converts text into TF-IDF weighted vectors
    TfidfVectorizer
)

from sklearn.metrics.pairwise import cosine_similarity
# Measures similarity between vector representations

# =========================
# Hugging Face Transformers
# =========================

from transformers import AutoTokenizer
# Generic tokenizer loader for transformer models

# =========================
# Levenshtein distance library
# =========================

import Levenshtein as lev

# Computes edit distance between strings
# Useful for spelling correction and fuzzy matching

# =========================
# Final confirmation message
# =========================

print("All imports successful. You are ready!")

All imports successful. You are ready!


---
## Section 1: Tokenization 

### Concept
**Tokenization** splits raw text into tokens (words, sub-words, or characters). It is the first step of almost every NLP pipeline.

| Method | Works by | Handles contractions (shortened forms of words or word combinations)? | Handles URLs? |
|--------|----------|-----------------------|---------------|
| Whitespace | str.split() | No | No |
| Regex | hand-written pattern | Partial | Partial |
| NLTK | trained on Penn Treebank | Yes | Partial |
| spaCy | rule + ML hybrid | Yes | Yes |
| BPE / WordPiece | learned merge rules | Yes | Yes |
---
> **Key insight:** There is no universally "best" tokenizer. The right choice depends on your language, domain, and downstream task.


In [25]:
# STARTER CODE -- Tokenization Method Comparison
# ----------------------------------------------
# This cell compares how different tokenisation methods behave on challenging inputs.
# Each sentence is designed to expose specific weaknesses in tokenisation strategies.

sample_texts = [
    # Sentence 1: abbreviations, contractions, and punctuation complexity
    "Dr. Smith couldn't attend the U.N. meeting -- it's a shame!",

    # Sentence 2: social media text with hashtags, mentions, and URL
    "I love NLP!!! #machinelearning @OpenAI https://huggingface.co",

    # Sentence 3: numeric formats, currency symbols, and date formatting
    "The price is $4.99 and the date is 22/05/2025.",
]

# -------------------------------------------------
# Method 1: Naive whitespace tokenisation
# -------------------------------------------------
# Splits text purely on whitespace characters.
# Problem: "Smith" and "Smith," become different tokens; punctuation stays attached.
def whitespace_tokenize(text):
    return text.split()

# -------------------------------------------------
# Method 2: Regex-based tokenisation
# -------------------------------------------------
# Uses a handcrafted pattern to capture words and numbers.
# Limitation: does not robustly handle URLs, hashtags, or social media syntax.
def regex_tokenize(text):
    # Regular expression for simple token extraction
    pattern = r"\b[a-zA-Z']+\b|\d+(?:\.\d+)?"

# ------------------------------------------------------------
# Breakdown of the expression: 
# This regex acts as a lightweight tokenizer that extracts:
#   - English word tokens (including contractions)
#   - numeric tokens (integers and decimals)
# ------------------------------------------------------------
# The pattern consists of two alternative matching rules separated by '|'
# meaning: match either a word-like token OR a numeric token.
#
# 1. \b[a-zA-Z']+\b
#    - \b              : word boundary  to ensure matching whole tokens only
#                       (prevents partial matches inside longer strings)
#    - [a-zA-Z']+      : one or more characters that are either:
#                        - uppercase letters A–Z
#                        - lowercase letters a–z
#                        - apostrophe (') to preserve contractions
#                          e.g. "don't", "it's", "couldn't"
#    - \b              : closing word boundary
# ------------------------------------------------------------
#
# 2. \d+(?:\.\d+)?
#    - \d+             : matches one or more digits (integer part)
#    - (?: ... )       : non-capturing group (groups pattern without storing it)
#    - \.              : literal dot character (escaped because '.' is special in regex)
#    - \d+             : one or more digits after decimal point
#    - ?               : makes the decimal portion optional
# ------------------------------------------------------------
    return re.findall(pattern, text)

# -------------------------------------------------
# Method 3: NLTK tokenisation
# -------------------------------------------------
# Uses a pre-trained rule-based tokenizer (Penn Treebank conventions).
# Strength: handles contractions more appropriately than simple heuristics. 
# knows "U.N." is an abbreviation, not a sentence end.
def nltk_tokenize(text):
    return word_tokenize(text)

# -------------------------------------------------
# Method 4: spaCy tokenisation
# -------------------------------------------------
# Uses a hybrid system combining linguistic rules and statistical models.
# Produces context-aware token boundaries and preserves structured entities.
def spacy_tokenize(text):
    # preloaded spaCy model, check Cell B
    doc = nlp(text)
    # list comprehension: iterates over doc and extracts token.text for each token.
    return [token.text for token in doc]

# -------------------------------------------------
# Method 5: GPT-2 Byte Pair Encoding (BPE)
# -------------------------------------------------
# Sub-word tokenisation based on learned merges from large corpora.
# Byte-Pair Encoding -- learned sub-word units. # The "Ġ" prefix (with a space encoded) marks the start of a new word.
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

def bpe_tokenize(text):
    return gpt2_tok.tokenize(text)

# -------------------------------------------------
# Dictionary of tokenisation methods for comparison
# -------------------------------------------------
methods = {
    "Whitespace": whitespace_tokenize,
    "Regex": regex_tokenize,
    "NLTK": nltk_tokenize,
    "spaCy": spacy_tokenize,
    "GPT-2 BPE": bpe_tokenize,
}

# -------------------------------------------------
# Execution loop: apply each tokenizer to each sample text
# -------------------------------------------------
for i, text in enumerate(sample_texts):
     # Improves readability by visually distinguishing different sections of output in a structured way.
    print(f"\n{'='*72}")
    print(f"Text {i+1}: {text}")
    print(f"{'='*72}")

    # Apply each tokenisation method and display results
    for name, fn in methods.items():

        tokens = fn(text)  # generate token list

        # print method name, token count, and token list
        print(f"[{name:7s}] ({len(tokens):2d} tokens)  {tokens}")


Text 1: Dr. Smith couldn't attend the U.N. meeting -- it's a shame!
[Whitespace] (11 tokens)  ['Dr.', 'Smith', "couldn't", 'attend', 'the', 'U.N.', 'meeting', '--', "it's", 'a', 'shame!']
[Regex  ] (11 tokens)  ['Dr', 'Smith', "couldn't", 'attend', 'the', 'U', 'N', 'meeting', "it's", 'a', 'shame']
[NLTK   ] (14 tokens)  ['Dr.', 'Smith', 'could', "n't", 'attend', 'the', 'U.N.', 'meeting', '--', 'it', "'s", 'a', 'shame', '!']
[spaCy  ] (14 tokens)  ['Dr.', 'Smith', 'could', "n't", 'attend', 'the', 'U.N.', 'meeting', '--', 'it', "'s", 'a', 'shame', '!']
[GPT-2 BPE] (18 tokens)  ['Dr', '.', 'ĠSmith', 'Ġcouldn', "'t", 'Ġattend', 'Ġthe', 'ĠU', '.', 'N', '.', 'Ġmeeting', 'Ġ--', 'Ġit', "'s", 'Ġa', 'Ġshame', '!']

Text 2: I love NLP!!! #machinelearning @OpenAI https://huggingface.co
[Whitespace] ( 6 tokens)  ['I', 'love', 'NLP!!!', '#machinelearning', '@OpenAI', 'https://huggingface.co']
[Regex  ] ( 8 tokens)  ['I', 'love', 'NLP', 'machinelearning', 'OpenAI', 'https', 'huggingface', 'co']
[NLT

### Reflection Questions:
Take **2 minutes** to note your observations **before moving on**. ✍️ 
Provide your answers here to the following questions:
- Which methods split "couldn't" correctly? Which keep "U.N." as one token?
- Which method keeps #machinelearning and https://... as single tokens?
- How does each method handle $4.99? Is the dollar sign included?


In [26]:
# EXERCISE 1.1 🟢 - Sentence Tokenization ✍️
# -----------------------------------------------
# TASK: Use NLTK's sent_tokenize() to split `paragraph` into sentences.
#       Print each sentence with its number, then print the total count.
#
# HINT:
#   sentences = sent_tokenize(paragraph)
#   for i, s in enumerate(sentences, start=1):
#       print(f"Sentence {i}: {s.strip()}")
#
# EXPECTED OUTPUT: 5 sentences.
# Notice that "Dr. Alan Turing" does NOT create a false sentence boundary --
# NLTK knows "Dr." is an abbreviation.

paragraph = """
Natural language processing is a subfield of linguistics and AI.
It focuses on the interactions between computers and human language.
NLP tasks include translation, summarization, and sentiment analysis.
Dr. Alan Turing proposed the Turing Test in 1950. Today, LLMs like GPT-4 pass it.
"""
# ✍️ YOUR CODE HERE
# Step 1 -- tokenize into sentences
sentences = None   # Replace with sent_tokenize(paragraph)

# Step 2 -- print results
# Uncomment after completing Step 1:
# for i, sent in enumerate(sentences, start=1):
#     print(f"Sentence {i}: {sent.strip()}")
# print(f"\nTotal: {len(sentences)} sentences")


In [27]:
# EXERCISE 1.2 🟡 - Social-Media Regex Tokenizer ✍️
# -------------------------------------------------------
# Social-media text breaks standard tokenizers:
#   - hashtags (#nlp), mentions (@user), URLs (https://...), emojis
#
# TASK: Complete the regex pattern inside social_media_tokenize() so that
#       it returns hashtags, mentions, URLs, and regular words as separate tokens.
#
# HINTS -- build your pattern from these pieces joined with "|":
#   URLs:      https?://\S+
#   Hashtags:  #\w+
#   Mentions:  @\w+
#   Words:     \b\w+\b
# Order matters! Put the longest/most-specific pattern first.

tweet = "Loving #NLP today @Stanford! Check https://arxiv.org for the latest papers"
# ✍️ YOUR CODE HERE
def social_media_tokenize(text):
    # Build your pattern here (join pieces with |)
    pattern = r""   # Fill this in
    return re.findall(pattern, text)

tokens = social_media_tokenize(tweet)
print("Input :", tweet)
print("Tokens:", tokens)
print("Expected: ['#NLP', '@Stanford', 'https://arxiv.org', 'Loving', ...]")


Input : Loving #NLP today @Stanford! Check https://arxiv.org for the latest papers
Tokens: ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
Expected: ['#NLP', '@Stanford', 'https://arxiv.org', 'Loving', ...]


In [28]:
# CHALLENGE 1.3 🔴 -Subword Tokenization: BERT vs GPT-2 ✍️
# -----------------------------------------------------------
# Sub-word tokenizers break rare/long words into learned pieces.
# BERT uses WordPiece (## marks a continuation piece).
# GPT-2 uses BPE (the G-with-space marks a word start).
#
# TASK: Run both tokenizers on the medical sentence.
#       Answer the **three reflection questions as comments below**.
# No code to write -- **focus on UNDERSTANDING the output.**

bert_tok     = AutoTokenizer.from_pretrained("bert-base-uncased")
medical_text = "The patient has hepatosplenomegaly and thrombocytopenia."

bert_tokens  = bert_tok.tokenize(medical_text)
gpt2_tokens  = gpt2_tok.tokenize(medical_text)

print("Input:", medical_text)
print(f"\nBERT  ({len(bert_tokens):2d} tokens): {bert_tokens}")
print(f"GPT-2 ({len(gpt2_tokens):2d} tokens): {gpt2_tokens}")

# ✍️ YOUR ANSWER HERE
# Reflection -- answer as comments:
# Q1. BERT uses "##" for continuation pieces (e.g. "##megaly").
#     Why is this marker necessary to reconstruct the original word?
# A1: 
#
# Q2. Which tokenizer produces fewer pieces for "hepatosplenomegaly"?
#     Does fewer always mean better for downstream tasks?
# A2:
#
# Q3. If you fine-tune BERT on medical text, does the sub-word vocabulary
#     adapt, or is it fixed from pre-training?
# A3:


Input: The patient has hepatosplenomegaly and thrombocytopenia.

BERT  (19 tokens): ['the', 'patient', 'has', 'he', '##pa', '##tos', '##ple', '##no', '##me', '##gal', '##y', 'and', 'th', '##rom', '##bo', '##cy', '##top', '##enia', '.']
GPT-2 (19 tokens): ['The', 'Ġpatient', 'Ġhas', 'Ġhepat', 'os', 'pl', 'en', 'ome', 'g', 'aly', 'Ġand', 'Ġth', 'rom', 'b', 'ocy', 'top', 'en', 'ia', '.']


---
## Section 2: Stemming & Lemmatization

### Concept
Both techniques normalise words to a common root so that "run", "running", and "ran" are all treated the same.

| Technique | How it works | Output of "running" | Always a real word? |
|-----------|-------------|---------------------|---------------------|
| Porter Stemmer | strip suffixes with heuristic rules | run | No ("studi" from "studies") |
| Snowball Stemmer | improved rule set | run | No |
| WordNet Lemmatizer | dictionary lookup + POS | run | Yes |
| spaCy Lemmatizer | neural POS + dictionary | run | Yes |
---

> **Rule of thumb:** 
> 
> 
- Use **stemming** when speed matters and approximate matching is OK **(e.g., search index)**. 
- Use **lemmatization** when linguistic correctness matters **(e.g., generation, QA)**.


In [29]:
# STARTER CODE- Stemmer & Lemmatizer side-by-side Table
# ------------------------------------------------
# Create a Porter Stemmer object
# Applies rule-based stemming to reduce words to their root form
porter = PorterStemmer()

# Create a Snowball Stemmer object for English
# Similar to Porter but generally more consistent and improved
snowball = SnowballStemmer("english")

# Create a WordNet Lemmatizer object
# Reduces words to their dictionary base form (lemma)
lemmatize = WordNetLemmatizer()

# List of example words for comparing stemming and lemmatisation
words = [
    "running", "runs", "ran",    # Different verb forms
    "studies", "studying",       # Verb and noun-related forms
    "better", "good", "best",    # Irregular adjective forms
    "universities",              # Plural noun
    "happily", "happiness",      # Words with derivational morphology
]

# Create a formatted table header
header = f"{'Word':<15} {'Porter':<12} {'Snowball':<12} {'Lemma(v)':<14} {'Lemma(n)'}"

# Print the table header
print(header)

# Print a separator line
print("-" * len(header))

# Iterate through each word in the list
for word in words:

    # Apply Porter stemming
    p = porter.stem(word)

    # Apply Snowball stemming
    s = snowball.stem(word)

    # Lemmatise the word assuming it is a verb
    # Example: "running" → "run"
    lv = lemmatize.lemmatize(word, pos="v")

    # Lemmatise the word assuming it is a noun
    # Example: "universities" → "university"
    ln = lemmatize.lemmatize(word, pos="n")

    # Print the original word and all processed outputs in aligned columns
    print(f"{word:<15} {p:<12} {s:<12} {lv:<14} {ln}")

Word            Porter       Snowball     Lemma(v)       Lemma(n)
-----------------------------------------------------------------
running         run          run          run            running
runs            run          run          run            run
ran             ran          ran          run            ran
studies         studi        studi        study          study
studying        studi        studi        study          studying
better          better       better       better         better
good            good         good         good           good
best            best         best         best           best
universities    univers      univers      universities   university
happily         happili      happili      happily        happily
happiness       happi        happi        happiness      happiness


In [30]:
# EXERCISE 2.1 🟢 - Normalize a Product Review ✍️
# ---------------------------------------------------
# TASK:
#   1. Tokenize `review` (lowercase)
#   2. Build `stems`  by applying porter.stem() to each token
#   3. Build `lemmas` by applying lemmatize.lemmatize() to each token
#   4. Print both lists and compare the number of UNIQUE forms
#
# STEP-BY-STEP HINTS:
#   tokens = word_tokenize(review.lower())
#   stems  = [porter.stem(t) for t in tokens]         <- list comprehension
#   lemmas = [lemmatize.lemmatize(t) for t in tokens] <- same pattern
#   Unique count: len(set(stems))

review = "The students were studying really hard because their studies matter for their future careers."

# ✍️ YOUR CODE HERE
# Step 1: Tokenize
tokens = word_tokenize(review.lower())
print("Tokens  :", tokens)

# Step 2: Stemming
stems  = []   # Replace with list comprehension using porter.stem()

# Step 3: Lemmatization
lemmas = []   # Replace with list comprehension using lemmatize.lemmatize()

# Step 4: Compare
print("\nStems   :", stems)
print("Lemmas  :", lemmas)
print(f"\nUnique stems  : {len(set(stems))}")
print(f"Unique lemmas : {len(set(lemmas))}")

# ✍️ YOUR ANSWER HERE
# Reflection (answer as a comment):
# Which has fewer unique forms?
# Does that make it "better"?
# When might stemming produce unhelpful results for this sentence?
# Answer:


Tokens  : ['the', 'students', 'were', 'studying', 'really', 'hard', 'because', 'their', 'studies', 'matter', 'for', 'their', 'future', 'careers', '.']

Stems   : []
Lemmas  : []

Unique stems  : 0
Unique lemmas : 0


In [31]:
# EXERCISE 2.2 🟡 - spaCy POS-Aware Lemmatization ✍️
# -------------------------------------------------------
# spaCy auto-detects part-of-speech (POS), so its lemmatizer knows
# whether "ran" is a VERB (-> "run") or hypothetically a NOUN.
# NLTK needs you to pass pos="v" or pos="n" manually.
#
# TASK: For every token in `sentence`:
#   - assign spacy_lemma using  token.lemma_
#   - assign pos             using  token.pos_    (coarse tag: NOUN, VERB, ...)
#   - keep   nltk_lemma as provided (no POS hint -- see what happens)
#   - print all four columns
#
# HINT: replace the two "???" strings with the correct token attributes.

sentence = "The geese were flying south while the mice ran through the better fields."
doc = nlp(sentence)

print(f"{'Token':<15} {'spaCy Lemma':<15} {'POS':<10} {'NLTK Lemma (no POS)'}")
print("-" * 62)
# ✍️ YOUR CODE HERE
for token in doc:
    spacy_lemma = "???"                                    # Use token.lemma_
    pos         = "???"                                    # Use token.pos_
    nltk_lemma  = lemmatize.lemmatize(token.text.lower()) # no POS hint
    print(f"{token.text:<15} {spacy_lemma:<15} {pos:<10} {nltk_lemma}")

# ✍️ YOUR ANSWER HERE
# Reflection:
# Look at "geese", "mice", "running", "ran".
# Where does NLTK (without POS) get the lemma wrong?
# Answer:


Token           spaCy Lemma     POS        NLTK Lemma (no POS)
--------------------------------------------------------------
The             ???             ???        the
geese           ???             ???        goose
were            ???             ???        were
flying          ???             ???        flying
south           ???             ???        south
while           ???             ???        while
the             ???             ???        the
mice            ???             ???        mouse
ran             ???             ???        ran
through         ???             ???        through
the             ???             ???        the
better          ???             ???        better
fields          ???             ???        field
.               ???             ???        .


---
## Section 3: Stop Words, Punctuation & Case Normalization

### Concept
**Stop words** are high-frequency, low-information words ("the", "is", "at"). 
Removing them reduces noise for classification and retrieval.

```
Raw Text -> lowercase -> remove URLs -> remove punctuation -> tokenize -> remove stop words -> cleaned tokens
```

> **Important Note:** Removing stop words is NOT always correct.
> - Sentiment analysis needs "not", "never": removing them flips meaning!
> - Named entity recognition needs "The" to detect "The Times".
> **Always** check your specific task *before removing anything*.


In [32]:
# STARTER CODE- Five-step Text cleaning Pipeline
# ------------------------------------------------
# Load the English stop-word list from NLTK
# Stop words are common words such as "the", "is", and "and"
STOP_WORDS = set(nltk_stopwords.words("english"))

# Print the total number of stop words
print(f"NLTK stop-word list: {len(STOP_WORDS)} words")

# Display the first 20 stop words alphabetically
print("First 20:", sorted(STOP_WORDS)[:20])

def clean_text(text, remove_stops=True, verbose=False):
    """
    A reusable five-step text-cleaning pipeline.

    Parameters
    ----------
    text         : raw input text
    remove_stops : remove stop words if True
    verbose      : display intermediate cleaning steps if True
    """

    # Print the original text before processing
    if verbose:
        print(f"[0] original  : {text}")

    # Step 1 -- convert all text to lowercase
    # This ensures that "NLP" and "nlp" are treated as the same token
    text = text.lower()

    if verbose:
        print(f"[1] lowercase : {text}")

    # Step 2 -- remove URLs using regular expressions
    # Removes links beginning with http, https, or www
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    if verbose:
        print(f"[2] no URLs   : {text}")

    # Step 3 -- remove punctuation characters
    # [^\w\s] matches anything that is NOT:
    #   \w -> a word character
    #   \s -> whitespace
    text = re.sub(r"[^\w\s]", "", text)

    if verbose:
        print(f"[3] no punct  : {text}")

    # Step 4 -- tokenize the cleaned text
    # Splits the sentence into individual words/tokens
    tokens = word_tokenize(text)

    if verbose:
        print(f"[4] tokenized : {tokens}")

    # Step 5 -- remove stop words and one-character tokens
    if remove_stops:

        # Keep tokens only if:
        #   1. the token is not a stop word
        #   2. the token length is greater than 1
        tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]

        if verbose:
            print(f"[5] no stops  : {tokens}")

    # Return the final cleaned tokens
    return tokens

# Example sentence for demonstrating the cleaning pipeline
demo = "The NLP model at https://huggingface.co is NOT working correctly!!!"

# Run the cleaning function with verbose output enabled
# This prints the result after every cleaning step
result = clean_text(demo, verbose=True)

# Print the final cleaned tokens
print(f"\nFinal tokens: {result}")

# Important observation:
# The word "not" is removed because it exists in the stop-word list.
# Removing negation words may change the sentiment or meaning of a sentence.
print("\nNotice: 'not' was removed -- that could flip the sentiment meaning!")

NLTK stop-word list: 198 words
First 20: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']
[0] original  : The NLP model at https://huggingface.co is NOT working correctly!!!
[1] lowercase : the nlp model at https://huggingface.co is not working correctly!!!
[2] no URLs   : the nlp model at  is not working correctly!!!
[3] no punct  : the nlp model at  is not working correctly
[4] tokenized : ['the', 'nlp', 'model', 'at', 'is', 'not', 'working', 'correctly']
[5] no stops  : ['nlp', 'model', 'working', 'correctly']

Final tokens: ['nlp', 'model', 'working', 'correctly']

Notice: 'not' was removed -- that could flip the sentiment meaning!


In [33]:
# EXERCISE 3.1 🟢 - Apply the pipeline to a corpus✍️
# -------------------------------------------------------
# TASK: For each sentence in `corpus`:
#   1. Count tokens BEFORE cleaning  -> raw_n
#   2. Count tokens AFTER  cleaning  -> clean_n
#   3. Compute the reduction %
#   4. Print a summary table + the cleaned tokens
#
# HINTS:
#   raw_n        = len(word_tokenize(doc))
#   clean_tokens = clean_text(doc)        <- no verbose needed here
#   clean_n      = len(clean_tokens)
#   pct          = (1 - clean_n / raw_n) * 100

corpus = [
    "I absolutely LOVE this product! It's amazing and works perfectly.",
    "This is the worst thing I've ever bought. Do NOT recommend it at all!",
    "Okay product, nothing special. The delivery was fast. Visit www.shop.com",
    "Can't believe how good this is!!! Best purchase of 2025... 10/10 would buy again.",
]

print(f"{'#':<3} {'Raw':>5} {'Clean':>6} {'Reduction':>10}   Cleaned tokens")
print("-" * 70)

# ✍️ YOUR CODE HERE
for i, doc in enumerate(corpus, start=1):
    raw_n        = len(word_tokenize(doc))

    # Step 1: Clean the document
    clean_tokens = []   # Replace with clean_text(doc)

    # Step 2: Compute reduction
    clean_n = len(clean_tokens)
    pct     = 0         # Replace with (1 - clean_n/raw_n)*100 if raw_n else 0

    print(f"{i:<3} {raw_n:>5} {clean_n:>6} {pct:>9.0f}%   {clean_tokens}")

# ✍️ YOUR ANSWER HERE
# Reflection: Which document has the highest reduction?
# Is that a problem for sentiment analysis?  Why?
# Answer:


#     Raw  Clean  Reduction   Cleaned tokens
----------------------------------------------------------------------
1      13      0         0%   []
2      17      0         0%   []
3      13      0         0%   []
4      20      0         0%   []


In [34]:
# EXERCISE 3.2 🟡 - Domain-Specific Stop Words✍️
# ----------------------------------------------------
# In a medical corpus, words like "patient" and "doctor" appear in
# EVERY note and add no information -- they become domain stop words.
#
# TASK:
#   1. Add at least 5 more medical domain stop words to `medical_stops`
#   2. For each note, tokenize (lower-cased) then filter using medical_stops
#   3. Print original + filtered
#
# HINT: filtered = [t for t in tokens if t not in medical_stops and len(t) > 1]

# ✍️ YOUR CODE HERE
medical_stops = STOP_WORDS | {
    "patient", "doctor",    # already provided
    # Add more domain stop words here, e.g.:
    # "prescribed", "reported", "noted", "presented", "admitted",
}

medical_notes = [
    "The patient presented with fever. The doctor prescribed antibiotics.",
    "Patient reported chest pain. Doctor ordered an ECG and blood tests.",
    "The patient was discharged. The doctor wrote a follow-up prescription.",
]

for note in medical_notes:
    tokens   = word_tokenize(note.lower())
    filtered = []   # Replace with the filtering list comprehension
    print(f"Original : {note}")
    print(f"Filtered : {filtered}\n")

# ✍️ YOUR ANSWER HERE
# Reflection: What key clinical information remains?
# What do you still lose?  How would you improve the filter?
# Answer:


Original : The patient presented with fever. The doctor prescribed antibiotics.
Filtered : []

Original : Patient reported chest pain. Doctor ordered an ECG and blood tests.
Filtered : []

Original : The patient was discharged. The doctor wrote a follow-up prescription.
Filtered : []



---
## Section 4: Bag of Words & CountVectorizer

### Concept
**Bag of Words (BoW)** represents a document as a count vector over a fixed vocabulary. Word ORDER is discarded, only **word frequencies matter**.

```
Vocab:          [cat, dog, likes, milk, runs]
"cat likes milk" -> [1, 0, 1, 1, 0]
"dog runs"       -> [0, 1, 0, 0, 1]
```

- **Limitations:** no semantics, no order, sparse & high-dimensional on large corpora.
- **Strengths:** fast, interpretable, surprisingly effective for many classification tasks.


In [35]:
# STARTER CODE- BoW with sklearn
# ------------------------------------------------
# Example text documents used for building the Bag-of-Words representation
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog are friends",
    "cats and dogs never agree",
]

# Create a CountVectorizer object from sklearn
# Automatically converts text into a BoW representation
cv = CountVectorizer()

# Learn the vocabulary and transform the corpus into a sparse matrix
# Sparse matrices save memory for large vocabularies
X = cv.fit_transform(corpus)

# Convert the sparse matrix into a dense NumPy array
# Then create a DataFrame for display
df_sk = pd.DataFrame(
    X.toarray(),
    columns=cv.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

# Display the sklearn-generated BoW matrix
print("\nsklearn CountVectorizer matrix:")
print(df_sk.to_string())


sklearn CountVectorizer matrix:
       agree  and  are  cat  cats  dog  dogs  friends  log  mat  never  on  sat  the
Doc 1      0    0    0    1     0    0     0        0    0    1      0   1    1    2
Doc 2      0    0    0    0     0    1     0        0    1    0      0   1    1    2
Doc 3      0    1    1    1     0    1     0        1    0    0      0   0    0    2
Doc 4      1    1    0    0     1    0     1        0    0    0      1   0    0    0


In [36]:
# EXERCISE 4.1 🟢 - Bigram CountVectorizer ✍️
# -----------------------------------------------
# A bigram captures TWO consecutive words.
# "not good" as a bigram is VERY different from its individual unigrams.
#
# TASK:
#   1. Create bi_vec = CountVectorizer(ngram_range=(2, 2))
#   2. Fit & transform sentiment_docs  ->  X_bi
#   3. Uncomment the print line to see bigram features
#   4. Answer the reflection question below
#
# HINT: the only change from uni_vec is the ngram_range argument.

sentiment_docs = [
    "the movie was not good at all",
    "the movie was very good indeed",
    "not bad at all for a cheap movie",
]

# Unigram (already done for you)
uni_vec = CountVectorizer(ngram_range=(1, 1))
X_uni   = uni_vec.fit_transform(sentiment_docs)
print("Unigram features:", uni_vec.get_feature_names_out().tolist())
# ✍️ YOUR CODE HERE
# Step 1: Create bigram vectorizer
bi_vec = None   # Replace: CountVectorizer(ngram_range=(2, 2))

# Step 2: Fit & transform
X_bi   = None   # Replace: bi_vec.fit_transform(sentiment_docs)

# Step 3: Print bigram features
# Uncomment once bi_vec is created:
# print("Bigram features:", bi_vec.get_feature_names_out().tolist())

# ✍️ YOUR ANSWER HERE
# Reflection:
# Find the bigrams "not good" and "very good" in the output.
# Using only unigrams, can a classifier tell the two sentences apart?
# With bigrams, can it?  Why?
# Answer:


Unigram features: ['all', 'at', 'bad', 'cheap', 'for', 'good', 'indeed', 'movie', 'not', 'the', 'very', 'was']


In [37]:
# EXERCISE 4.2 🟡 -- Term Frequency Analysis ✍️
# ------------------------------------------------
# TASK: Using CountVectorizer on news_corpus, answer three questions:
#   Q1. What are the top-10 most frequent terms overall?
#   Q2. Which terms appear in ALL 5 documents?
#   Q3. Which terms appear in exactly ONE document?
#
# Arrays `freq` (total count) and `doc_freq` (num docs with term)
# are already built. Your job is to query them.
#
# HINTS:
#   Q1: top10 = np.argsort(freq)[::-1][:10]
#       then loop over top10 to print feature_names[idx]
#   Q2: feature_names[doc_freq == len(news_corpus)]
#   Q3: feature_names[doc_freq == 1]

news_corpus = [
    "Scientists discover new AI model that beats humans at chess and go",
    "New AI research shows large language models can reason about math",
    "Humans and AI collaborate on scientific discovery in new study",
    "Chess champion defeats AI model in controversial match",
    "Language models show surprising reasoning about human emotions",
]

vec           = CountVectorizer(stop_words="english")
X             = vec.fit_transform(news_corpus)
feature_names = vec.get_feature_names_out()
freq          = X.toarray().sum(axis=0)           # total count per term
doc_freq      = (X.toarray() > 0).sum(axis=0)    # num docs containing term

# ✍️ YOUR CODE HERE
# Q1: Top-10 most frequent terms
print("Q1 - Top 10 most frequent terms:")
top10 = None   # np.argsort(freq)[::-1][:10]
# Uncomment to print:
# for idx in top10:
#     print(f"  {feature_names[idx]:<20} freq={freq[idx]}  doc_freq={doc_freq[idx]}")

# Q2: Terms in ALL documents
print("\nQ2 - Terms in ALL 5 documents (very common):")
# print(feature_names[doc_freq == len(news_corpus)])

# Q3: Terms in exactly ONE document
print("\nQ3 - Terms in exactly ONE document (very distinctive):")
# print(feature_names[doc_freq == 1])


# ✍️ YOUR ANSWER HERE
# Reflection:
# What does it tell you if a term has a high total frequency but appears in only a few documents?
# How does removing stop words change the terms you see in Q1, Q2, and Q3?
# Could term frequency alone be misleading in understanding document importance? Why or why not?
# Answer:

Q1 - Top 10 most frequent terms:

Q2 - Terms in ALL 5 documents (very common):

Q3 - Terms in exactly ONE document (very distinctive):


---
## Section 5: TF-IDF

### Concept
**TF-IDF** gives high weight to words that are **frequent in one document but rare across the corpus**; exactly the words that identify a document's unique topic.

```
Term Frequency (TF(t,d))= count(t in d) / total_words_in_d       <- how often in this doc
Inverse Document Frequency (IDF(t))= log(N / df(t))              <- how rare across all docs
Term Frequency-Inverse Document Frequency (TF-IDF(t, d))= TF(t, d) x IDF(t)
```

| Score | High value means |
|-------|------------------|
| TF | word appears many times in THIS document |
| IDF | word is RARE across the corpus |
| TF-IDF | word is locally frequent AND globally distinctive |


In [38]:
# STARTER CODE- sklearn TF-IDF
# sklearn applies smoothing (+1 to IDF numerator and denominator)
# ------------------------------------------------
docs = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks for machine learning",
    "natural language processing is a field of artificial intelligence",
    "neural networks are the backbone of deep learning models"
]
# Create a TF-IDF Vectorizer object
# TF-IDF measures how important a word is within a document
# relative to the entire collection of documents
tfidf_vec = TfidfVectorizer()

# Learn the vocabulary and transform the documents into a TF-IDF matrix
# The result is stored as a sparse matrix for memory efficiency
X_tfidf = tfidf_vec.fit_transform(docs)

# Convert the sparse matrix into a pandas DataFrame
# round(3) limits the values to 3 decimal places for readability
df_tfidf = pd.DataFrame(
    X_tfidf.toarray().round(3),

    # Extract the vocabulary terms used as column names
    columns=tfidf_vec.get_feature_names_out(),

    # Create row labels for each document
    index=[f"Doc {i+1}" for i in range(len(docs))]
)

# Display the TF-IDF matrix
print("sklearn TF-IDF matrix:")
print(df_tfidf.to_string())

# Print the top-3 highest TF-IDF terms for each document
# Higher TF-IDF values indicate more important words in that document
print("\nTop-3 most important terms per document:")

# Iterate through each document row
for i in range(len(docs)):

    # Select the three largest TF-IDF values in the current document
    top3 = df_tfidf.iloc[i].nlargest(3)

    # Print the term names and their TF-IDF scores
    print(f"  Doc {i+1}: {list(zip(top3.index, top3.round(3).values))}")


sklearn TF-IDF matrix:
         are  artificial  backbone   deep  field    for  intelligence     is  language  learning  machine  models  natural  networks  neural     of  processing  subset    the   uses
Doc 1  0.000       0.380     0.000  0.000  0.000  0.000         0.380  0.380     0.000     0.308    0.380   0.000    0.000     0.000   0.000  0.308       0.000   0.482  0.000  0.000
Doc 2  0.000       0.000     0.000  0.319  0.000  0.404         0.000  0.000     0.000     0.516    0.319   0.000    0.000     0.319   0.319  0.000       0.000   0.000  0.000  0.404
Doc 3  0.000       0.315     0.000  0.000  0.399  0.000         0.315  0.315     0.399     0.000    0.000   0.000    0.399     0.000   0.000  0.255       0.399   0.000  0.000  0.000
Doc 4  0.387       0.000     0.387  0.305  0.000  0.000         0.000  0.000     0.000     0.247    0.000   0.387    0.000     0.305   0.305  0.247       0.000   0.000  0.387  0.000

Top-3 most important terms per document:
  Doc 1: [('subset', np.f

In [39]:
# EXERCISE 5.1 🟡 -- TF-IDF Document Retrieval (mini search engine) ✍️
# -----------------------------------------------------------------------
# Cosine similarity between a query vector and document vectors is how
# classic search engines rank results.
#
# TASK -- complete the three marked steps:
#   Step 1: fit TfidfVectorizer on article_corpus
#   Step 2: transform each query into the SAME vector space
#   Step 3: compute cosine similarity and print the best match
#
# HINTS:
#   Step 1:  corpus_matrix = tfidf.fit_transform(article_corpus)
#   Step 2:  query_vec     = tfidf.transform([query])   <- transform ONLY, not fit!
#   Step 3:  scores        = cosine_similarity(query_vec, corpus_matrix)[0]
#            best_idx      = np.argmax(scores)
#
# IMPORTANT: always call .transform() (not .fit_transform()) on the query.
# Refitting would rebuild the vocabulary around just the query words,
# making the document vectors incompatible.

article_corpus = [
    "Python is a popular programming language used in data science and machine learning.",
    "Machine learning algorithms learn patterns from data without explicit programming.",
    "Natural language processing enables computers to understand human language.",
    "Deep learning neural networks have revolutionized computer vision and speech recognition.",
    "Data science combines statistics, programming, and domain expertise.",
]

queries = [
    "how do computers understand text?",
    "what programming language for data science?",
    "neural networks for images",
]

# ✍️ YOUR CODE HERE
# Step 1: Fit the vectorizer on the corpus
tfidf         = TfidfVectorizer(stop_words="english")
corpus_matrix = None   # Replace: tfidf.fit_transform(article_corpus)

# Steps 2 & 3: Search for each query
for query in queries:
    if corpus_matrix is None:
        print(f"Query: '{query}'  -> (complete Step 1 first)")
        continue
    # Uncomment and complete:
    # query_vec  = 
    # scores     = cosine_similarity(query_vec, corpus_matrix)[0]
    # best_idx   = 
    # print(f"Query : '{query}'")
    # print(f"Match : '{article_corpus[best_idx]}'  (score={scores[best_idx]:.3f})")
    # print()


Query: 'how do computers understand text?'  -> (complete Step 1 first)
Query: 'what programming language for data science?'  -> (complete Step 1 first)
Query: 'neural networks for images'  -> (complete Step 1 first)


---
## Section 6: String Similarity Measures

### Concept

| Measure | Level | What it counts | Best for | Scoring / Interpretation |
|---------|-------|----------------|----------|-------------------------|
| Edit Distance | character | minimum number of insert, delete, or replace operations needed to transform one string into another | spell-check, fuzzy match | **Lower values indicate higher similarity**. A score of 0 means the strings are identical; larger values indicate more dissimilarity. |
| Jaccard | word-set | number of shared words divided by total unique words across both strings | short-text deduplication | **Values range from 0 to 1**. Higher values indicate greater similarity (1 = identical sets, 0 = no overlap). |
| Cosine | TF-IDF vector | angle between document vectors in high-dimensional space | document similarity | **Values range from 0 to 1**. Higher values indicate higher similarity (1 = vectors point in the same direction, 0 = vectors are orthogonal). |

### How Scoring Works

1. **Edit Distance**: Compares strings character by character. Each insertion, deletion, or substitution increases the distance. It is sensitive to small changes and useful for detecting typos.  

2. **Jaccard Index**: Treats texts as sets of words. Only the presence or absence of words matters, not their order or frequency. Useful when comparing short documents or phrases.  

3. **Cosine Similarity**: Uses vector representations (often TF-IDF) to capture both presence and importance of terms. It measures the cosine of the angle between two vectors: closer angles mean higher similarity. Unlike edit distance, it handles longer documents well and is less sensitive to exact word order.  

In [40]:
# STARTER CODE -- Three Similarity Functions + Comparison Table
# -------------------------------------------------------------
# All three functions are fully implemented below.
# Read each one, then run the comparison table and study the results.
# -------------------------------------------------------------

# 1. Levenshtein edit distance
# Computes the Levenshtein edit distance between two input strings
# by delegating the computation to the Levenshtein library implementation
def edit_dist(s1, s2):
    return lev.distance(s1, s2)

# 2. Jaccard similarity (token-level overlap)
# Measures similarity between two sets of words
def jaccard(s1, s2):
    """
    Computes intersection-over-union of token sets.
    """

    # Convert both strings into sets of lowercase words
    a, b = set(s1.lower().split()), set(s2.lower().split())

    # Jaccard similarity = |A ∩ B| / |A ∪ B|
    return len(a & b) / len(a | b) if a | b else 0


# 3. Cosine similarity (TF-IDF based)
# Measures similarity between vectorised text representations
def cosine_sim(s1, s2):
    """
    Builds TF-IDF vectors for two strings and computes cosine similarity.
    """

    # Create TF-IDF vectoriser
    v = TfidfVectorizer()

    # Transform the two input strings into TF-IDF vectors
    mat = v.fit_transform([s1, s2])

    # Compute cosine similarity between the two vectors
    return cosine_similarity(mat[0], mat[1])[0][0]


# Define pairs of sentences for comparison
pairs = [
    ("kitten", "sitting"),
    ("hello world", "hello NLP"),
    ("I love NLP", "NLP I love"),       # same words, different order
    ("data science", "data engineering"),
    ("the quick brown fox", "the slow green cat"),
]

# Print table header
print(f"{'Pair':<44} {'Edit':>6} {'Jaccard':>8} {'Cosine':>8}")
print("-" * 72)

# Compute and display similarity metrics for each pair
for s1, s2 in pairs:

    # Create a readable label for the pair
    label = f'"{s1}" vs "{s2}"'

    # Compute and print all three similarity measures
    print(
        f"{label:<44} "
        f"{edit_dist(s1, s2):>6} "
        f"{jaccard(s1, s2):>8.3f} "
        f"{cosine_sim(s1, s2):>8.3f}"
    )


# ✍️ YOUR ANSWER HERE
# Observation: "I love NLP" vs "NLP I love"
# Why is Cosine = 1.0 but Edit distance is large?
# Answer:


Pair                                           Edit  Jaccard   Cosine
------------------------------------------------------------------------
"kitten" vs "sitting"                             3    0.000    0.000
"hello world" vs "hello NLP"                      5    0.333    0.336
"I love NLP" vs "NLP I love"                      8    1.000    1.000
"data science" vs "data engineering"              9    0.333    0.336
"the quick brown fox" vs "the slow green cat"     11    0.143    0.144


In [41]:
# EXERCISE 6.1 🟢 -Spelling Corrector ✍️
# ------------------------------------------
# TASK: Implement find_closest() using the edit_dist() function above.
#       For each misspelled word, find the dictionary word with
#       the smallest edit distance.
#
# HINT: return min(dictionary, key=lambda w: edit_dist(word, w))
#Before:-
# NLP_DICT = [
#     "natural","language","processing","machine","learning",
#     "algorithm","tokenization","lemmatization","embedding","transformer",
#     "attention","neural","network","classification","regression",
# ]

# misspelled = ["natral","laguage","procssing","alogithm","tokeniaztion"]
# # ✍️ YOUR CODE HERE
# def find_closest(word, dictionary):
#     # Replace with one-line min() call
#     return None

# for mw in misspelled:
#     correction = find_closest(mw, NLP_DICT)
#     print(f"  '{mw}'  ->  '{correction}'")
# After:- 
NLP_DICT = [
    "natural", "language", "processing", "machine", "learning",
    "algorithm", "tokenization", "lemmatization", "embedding", "transformer",
    "attention", "neural", "network", "classification", "regression",
]

misspelled = ["natral", "laguage", "procssing", "alogithm", "tokeniaztion"]

def find_closest(word, dictionary):
    return min(dictionary, key=lambda w: edit_dist(word, w))

for mw in misspelled:
    correction = find_closest(mw, NLP_DICT)
    print(f"  '{mw}'  ->  '{correction}'")


  'natral'  ->  'natural'
  'laguage'  ->  'language'
  'procssing'  ->  'processing'
  'alogithm'  ->  'algorithm'
  'tokeniaztion'  ->  'tokenization'


In [42]:
# Exercise 6.2 🔴 -Near-Duplicate Review Detector **(BONUS)** ✍️
# -----------------------------------------------------
# Near-duplicate detection is critical for review platforms:
# bots sometimes post slightly paraphrased spam reviews.
#
# TASK:
#   1. For every pair (i, j) with i < j, compute Jaccard and Cosine
#   2. Flag the pair as near-duplicate if BOTH exceed THRESHOLD
#   3. Try different THRESHOLD values and observe what changes
#
# HINTS:
#   Loop: for i in range(len(reviews)):
#             for j in range(i+1, len(reviews)):
#   is_dup = "DUP" if jac > THRESHOLD and cos > THRESHOLD else ""
reviews = [
    "Great product, very happy with my purchase!",           # 0
    "Great product, I am very happy with this purchase!",   # 1  near-dup of 0
    "Terrible quality, waste of money. Would not buy again.",# 2
    "Awful quality, total waste of money. Never buying again.",# 3 near-dup of 2
    "Fast delivery and good packaging. Product works fine.", # 4
    "Quick shipping, nice box. Works as expected.",          # 5  near-dup of 4
]
# ✍️ YOUR CODE HERE
THRESHOLD = 0.4   # Try 0.3, 0.5, 0.6 and observe
def edit_dist(s1, s2):
    return lev.distance(s1, s2)

# 2. Jaccard similarity (token-level overlap)
# Measures similarity between two sets of words
def jaccard(s1, s2):
    """
    Computes intersection-over-union of token sets.
    """

    # Convert both strings into sets of lowercase words
    a, b = set(s1.lower().split()), set(s2.lower().split())

    # Jaccard similarity = |A ∩ B| / |A ∪ B|
    return len(a & b) / len(a | b) if a | b else 0


# 3. Cosine similarity (TF-IDF based)
# Measures similarity between vectorised text representations
def cosine_sim(s1, s2):
    """
    Builds TF-IDF vectors for two strings and computes cosine similarity.
    """

    # Create TF-IDF vectoriser
    v = TfidfVectorizer()

    # Transform the two input strings into TF-IDF vectors
    mat = v.fit_transform([s1, s2])

    # Compute cosine similarity between the two vectors
    return cosine_similarity(mat[0], mat[1])[0][0]
print(f"{'Pair':<8} {'Jaccard':>9} {'Cosine':>9} {'Flag':>12}")
print("-"*45)
# ✍️ YOUR CODE HERE
for i in range(len(reviews)):
    for j in range(i+1, len(reviews)):
        # Compute similarities
        jac    = jaccard(reviews[i], reviews[j])   # Replace
        cos    = cosine_sim(reviews[i], reviews[j])
        is_dup = "DUP" if jac > THRESHOLD and cos > THRESHOLD else ""
        print(f"({i},{j})   {jac:>9.3f} {cos:>9.3f} {is_dup:>12}")

# ✍️ YOUR ANSWER HERE
# Reflection Questions: 
# Which THRESHOLD catches all 3 true duplicates without flagging any false positives?
# How does changing the THRESHOLD value affect which review pairs are flagged as near-duplicates?  
# Is there a threshold that perfectly separates true duplicates from non-duplicates? Why or why not?
# Compare Jaccard and Cosine similarity scores for the same pair. Why might one score be higher than the other?  
# Which measure seems more sensitive to small changes in wording or word order?
# Are there review pairs that are flagged as duplicates but are actually not (false positives)?  
# Are there true duplicates that are missed at certain thresholds (false negatives)?  
# In a real review platform, what could be the consequences of setting the threshold too low or too high?  
# How might combining multiple similarity measures help reduce errors in detecting near-duplicates?
# How would you modify the approach if reviews were much longer, or if they contained synonyms and paraphrases?  
# Could adding preprocessing (like lemmatization or removing stop words) improve detection accuracy?  
# Answers:


Pair       Jaccard    Cosine         Flag
---------------------------------------------
(0,1)       0.600     0.674          DUP
(0,2)       0.000     0.000             
(0,3)       0.000     0.000             
(0,4)       0.000     0.072             
(0,5)       0.000     0.000             
(1,2)       0.000     0.000             
(1,3)       0.000     0.000             
(1,4)       0.000     0.067             
(1,5)       0.000     0.000             
(2,3)       0.385     0.388             
(2,4)       0.000     0.000             
(2,5)       0.000     0.000             
(3,4)       0.000     0.000             
(3,5)       0.000     0.000             


(4,5)       0.071     0.072             


---
## Lab Wrap-Up: Week 1

| Section | Key Concept | Remember |
|---------|------------|----------|
| 1 | Tokenization | No single "best" tokenizer; match method to task |
| 2 | Stemming vs Lemmatization | Stemming = fast & approximate; lemmatization = accurate |
| 3 | Stop words | Removing them is NOT always correct (sentiment!) |
| 4 | Bag of Words | Simple but effective; bigrams capture some word order |
| 5 | TF-IDF | Rewards locally-frequent AND globally-rare words |
| 6 | Similarity | Edit = char-level; Jaccard = set; Cosine = vector |

### Further Reading
- [NLTK Book (free)](https://www.nltk.org/book/) chapters 1-3
- [spaCy 101](https://spacy.io/usage/spacy-101)
- [HuggingFace Tokenizer docs](https://huggingface.co/docs/tokenizers)
- [scikit-learn text features](https://scikit-learn.org/stable/modules/feature_extraction.html)
- [Jurafsky & Martin ch. 2 (free PDF)](https://web.stanford.edu/~jurafsky/slp3/)